# 06 Broadcasting

| | |
|---|---|
| **Path** | NumPy |
| **Prerequisite** | 05 |

Broadcasting is the rule NumPy uses when shapes differ. Line the shapes up on the **right**. Walk from the right, one axis at a time.

1. If an axis is missing, treat its length as 1.
2. If the two lengths are equal, keep that length.
3. If one length is 1, stretch it to the other length.
4. Otherwise the operation is illegal.

Stretching does not copy the data in the expression you write. It reuses an element along that axis.

$$
\begin{array}{ccc}
\text{left} & \text{right} & \text{result} \\
(3,) & (3,) & (3,) \\
(3,1) & (1,4) & (3,4) \\
(3,1) & (4,) & (3,4) \\
(3,2) & (3,) & \text{illegal}
\end{array}
$$

The illegal row fails because the rightmost lengths are 2 and 3, and neither is 1.


## Example 1 — A column plus a row

### Predict

Shapes `(3, 1)` and `(1, 4)` line up as equal rank. The length-1 column stretches across four columns. The length-1 row stretches down three rows. Each sum is one column entry plus one row entry.

$$
\begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}
+
\begin{bmatrix} 10 & 20 & 30 & 40 \end{bmatrix}
=
\begin{bmatrix}
11 & 21 & 31 & 41 \\
12 & 22 & 32 & 42 \\
13 & 23 & 33 & 43
\end{bmatrix}
$$

### Run


In [1]:
# Example 1 — A column plus a row.
import numpy as np

column = np.array([[1], [2], [3]])
row = np.array([[10, 20, 30, 40]])
print(column.shape, row.shape)
total = column + row
print(total.shape)
print(total)


(3, 1) (1, 4)
(3, 4)
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


## Example 2 — A missing axis counts as 1

### Predict

A one-dimensional row of length 4 is treated as shape `(1, 4)` when it meets a column of shape `(3, 1)`. The result is the same matrix as example 1. The right-alignment is what makes `(4,)` match the last axis.

### Run


In [2]:
import numpy as np

column = np.array([[1], [2], [3]])
# Shape (4,) is aligned with the last axis, as if it were (1, 4).
row = np.array([10, 20, 30, 40])
print((column + row).shape)
print(column + row)


(3, 4)
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


## Example 3 — A multiplication table from two axes

### Predict

`np.arange(1, 5)` is $1,2,3,4$. As a column times a row, entry $(i, j)$ is the product of those two factors.

$$
\begin{bmatrix} 1\\2\\3\\4 \end{bmatrix}
\begin{bmatrix} 1 & 2 & 3 & 4 \end{bmatrix}
=
\begin{bmatrix}
1 & 2 & 3 & 4 \\
2 & 4 & 6 & 8 \\
3 & 6 & 9 & 12 \\
4 & 8 & 12 & 16
\end{bmatrix}
$$

### Run


In [3]:
import numpy as np

factors = np.arange(1, 5)
# None inserts an axis, turning shape (4,) into (4, 1).
table = factors[:, None] * factors[None, :]
print(table)


[[ 1  2  3  4]
 [ 2  4  6  8]
 [ 3  6  9 12]
 [ 4  8 12 16]]


## Example 4 — Centering columns, once the shapes can meet

### Predict

Column means of

$$
\begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}
$$

are $2$ and $3$. Those means have shape `(2,)`. Against a grid of shape `(2, 2)`, the length 2 lines up with the last axis, so the subtraction is legal. Each column loses its own mean:

$$
\begin{bmatrix} 1-2 & 2-3 \\ 3-2 & 4-3 \end{bmatrix}
=
\begin{bmatrix} -1 & -1 \\ 1 & 1 \end{bmatrix}.
$$

Lesson 07 names `axis` and `keepdims`. This example only checks that the means can meet the grid.

### Run


In [4]:
# Example 4 — Centering columns, once the shapes can meet.
import numpy as np

grid = np.array([[1.0, 2.0], [3.0, 4.0]])
means = grid.mean(axis=0)
print(means)
print(grid - means)


[2. 3.]
[[-1. -1.]
 [ 1.  1.]]


## A pitfall

`(3, 2) + (3,)` aligns `(3,)` with the last axis. The last axis has length 2, and 3 is not 2 and not 1. The shapes look related because both mention 3, but 3 is on the left of the larger shape. Right-alignment never sees it as a match.

### Run


In [5]:
import numpy as np

grid = np.zeros((3, 2))
extra = np.zeros(3)
try:
    print(grid + extra)
except ValueError as error:
    print(type(error).__name__)
# A length-2 vector matches the last axis.
print((grid + np.zeros(2)).shape)


ValueError
(3, 2)


## Summary

Align shapes on the right. Equal lengths stay, a length of 1 stretches, and a missing axis counts as 1. Anything else is a `ValueError`, even when a number you care about appears somewhere in both shapes. `[:, None]` and `[None, :]` are how you turn two vectors into the rows and columns of a table.
